# <center> **MACHINE LEARNING**

### **1) IMPORTS LIBRARIES AND DATASETS**

#### **a) Libraries**

In [4]:
# general imports 
import os
import pandas as pd
import numpy as np
import requests 
import time  

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
import pickle

#### **b) Datasets**

In [6]:
ratings = pd.read_parquet("C://Users/barba/Case_studies/Cinema_recommender/cleaned_data/ratings.parquet", engine='pyarrow')
movies = pd.read_parquet("C://Users/barba/Case_studies/Cinema_recommender/cleaned_data/movies.parquet", engine='pyarrow')
genres = pd.read_parquet("C://Users/barba/Case_studies/Cinema_recommender/cleaned_data/genres.parquet", engine='pyarrow')
directors = pd.read_parquet("C://Users/barba/Case_studies/Cinema_recommender/cleaned_data/directors.parquet", engine='pyarrow')
actors = pd.read_parquet("C://Users/barba/Case_studies/Cinema_recommender/cleaned_data/actors.parquet", engine='pyarrow')

#### **c) Merging tables**

In [8]:
# Creation of a meta_table by merging useful tables: 
meta_table = movies.copy()
meta_table = meta_table.merge(ratings, how='left', on='tconst')
meta_table = meta_table.merge(directors, how='left', on='tconst')
meta_table = meta_table.merge(actors, how= 'left', on='tconst' )
meta_table = meta_table.merge(genres, how='left', on='tconst' )

#### **d) Final table**

In [9]:
# renaming columns 
meta_table = meta_table.rename(columns={'primaryName_x': 'director_name',
                                        'primaryName_y': 'actor_name',
                                        'startYear': 'year'})
# drop some columns that add noise such as nconst, birth and death year, region, categories 
meta_table= meta_table[['tconst','primaryTitle', 'title', 'director_name','actor_name', 'genres', 'year', 'detected_language','runtimeMinutes', 'averageRating', 'numVotes']]

In [10]:
# CLEANING 
meta_table.drop_duplicates(inplace=True) # suppression of more than 100000 duplicates
meta_table.dropna(inplace=True) # suppression of null values for our machine learning algorithm 

In [11]:
meta_table.sample(n=10, random_state=42) # this is a sample of 10 rows from our final meta_table 

,tconst,primaryTitle,title,director_name,actor_name,genres,year,detected_language,runtimeMinutes,averageRating,numVotes
594565,tt5153236,Hampstead,Hampstead,Joel Hopkins,Brendan Gleeson,Comedy,2017,en,102,6.1,8243.0
535180,tt3129692,Stealing Chanel,Le voleur au grand cœur,Roberto Mitrotti,Anna Maria Cianciulli,Romance,2015,en,102,5.4,94.0
260101,tt0365478,Man with the Screaming Brain,Man with the Screaming Brain,Bruce Campbell,Jonas Talkington,Comedy,2005,en,90,5.4,5810.0
417930,tt1558741,Comme les cinq doigts de la main,Comme les cinq doigts de la main,Alexandre Arcady,Mathieu Delarive,Drama,2010,fr,116,5.3,611.0
23463,tt0085501,Erendira,Erendira,Ruy Guerra,Irene Papas,Drama,1983,fr,103,6.6,730.0
614423,tt6034966,The Lion Sleeps Tonight,Le lion est mort ce soir,Nobuhiro Suwa,Pauline Etienne,Drama,2017,fr,103,6.4,330.0
424426,tt1610525,Chuck,Outsider,Philippe Falardeau,Naomi Watts,Biography,2016,en,98,6.5,7355.0
266346,tt0383010,The Three Stooges,Les trois corniauds,Peter Farrelly,Will Sasso,Comedy,2012,en,92,5.2,34406.0
615620,tt6105098,The Lion King,Le Roi lion,Jon Favreau,JD McCrary,Drama,2019,en,118,6.8,288298.0
606280,tt5751998,Small Town Crime,Small Town Crime,Eshom Nelms,Jeremy Ratchford,Action,2017,en,91,6.6,13206.0


In [12]:
# 1. REMOVE DUPLICATES 
# We ensure one row per movie ID (tconst)
# keep='first' keeps the row with the most data if there are differences
unique_movies = meta_table.drop_duplicates(subset=['tconst'], keep='first')
print(f"✅ Duplicates removed. Final count: {unique_movies.shape[0]} unique movies.")

# 2. ADD POSTERS
# Define the function to fetch from Imdb
def get_static_poster(imdb_id):
    # 🔑 REPLACE WITH YOUR ACTUAL KEY
    api_key = "fc730df7fa4ba8dbf7ba09f1ceb0359c" 
    
    try:
        url = f"https://api.themoviedb.org/3/find/{imdb_id}?api_key={api_key}&external_source=imdb_id"
        response = requests.get(url, timeout=2).json()
        
        # Check if results exist
        if response.get('movie_results'):
            poster_path = response['movie_results'][0]['poster_path']
            return f"https://image.tmdb.org/t/p/w500{poster_path}"
        else:
            return None # Return None so we can use a placeholder later
            
    except Exception as e:
        return None

✅ Duplicates removed. Final count: 20222 unique movies.


In [ ]:
unique_movies['poster_url'] = unique_movies['tconst'].apply(get_static_poster)

In [ ]:
# Compression of meta_table
meta_table = meta_table.groupby('tconst').agg({
    'primaryTitle': 'first',      
    'title': 'first',             
    'year': 'first',              
    'averageRating': 'mean',      
    'numVotes': 'max',            
    'runtimeMinutes': 'first',         
    'detected_language': 'first', 
    
    # CRITICAL: Combine text features so no one gets deleted
    # We use set() to remove duplicates inside the string (e.g., "Brad Pitt, Brad Pitt")
    'actor_name': lambda x: ', '.join(sorted(set(','.join(x).split(',')))), 
    'director_name': lambda x: ', '.join(sorted(set(','.join(x).split(',')))),
    'genres': lambda x: ', '.join(sorted(set(','.join(x).split(',')))),
    
    # Re-create the soup from the newly compressed columns
    # (We will do this after the grouping)
}).reset_index()

In [ ]:
# include poster_url into meta_table 
meta_table = meta_table.merge(unique_movies[['tconst', 'poster_url']], how= 'left', on = 'tconst')

In [20]:
meta_table

,tconst,primaryTitle,title,year,averageRating,numVotes,runtimeMinutes,detected_language,actor_name,director_name,genres,poster_url
0,tt0035423,Kate & Leopold,Kate et Léopold,2001,6.4,93042.0,118,en,"Bradley Whitford, Breckin Meyer, Hugh Jackman,...",James Mangold,"Comedy, Fantasy, Romance",https://image.tmdb.org/t/p/w500/57kLEPlIXHynZX...
1,tt0036606,"Another Time, Another Place",Les Coeurs captifs,1983,6.4,378.0,118,en,"Claudio Rosini, Denise Coffey, Gianluca Favill...",Michael Radford,"Drama, War",https://image.tmdb.org/t/p/w500/anoPMnxdrL4B7E...
2,tt0048550,Rendez-vous of the Docks,Le rendez-vous des quais,1990,6.6,69.0,75,fr,"Albert Manach, André Maufray, Andrée Biancheri...",Paul Carpita,Drama,https://image.tmdb.org/t/p/w500/aV3fve65ivMtOj...
3,tt0065530,Le cercle des passions,Le cercle des passions,1983,6.4,79.0,108,fr,"Assumpta Serna, Dora Calindri, Federico Pacifi...",Claude d'Anna,Drama,https://image.tmdb.org/t/p/w500/wJ1IPFSRyJmHBG...
4,tt0067100,Firecracker,Attaque à mains nues,1981,5.6,1019.0,77,en,"Carolyn Smith, Chanda Romero, Darby Hinton, Do...",Cirio H. Santiago,"Action, Drama, Thriller",https://image.tmdb.org/t/p/w500/5wn5mEzpni6hMv...
...,...,...,...,...,...,...,...,...,...,...,...,...
20217,tt9896916,Pilgrim's Progress,Le Voyage du pèlerin,2019,6.5,1047.0,108,en,"Andy Harrison, David Thorpe, Jasmine Jones, Jo...",Robert Fernandez,"Adventure, Animation, Family",https://image.tmdb.org/t/p/w500/vtfsNxAsDHElFv...
20218,tt9898858,Coffee & Kareem,Coffee & Kareem,2020,5.2,15181.0,88,en,"Andrew Bachelor, Betty Gilpin, David Alan Grie...",Michael Dowse,"Action, Comedy, Crime",https://image.tmdb.org/t/p/w500/fsIIv6QKDlqrVy...
20219,tt9904530,Scream Returns,Scream Returns,2018,5.7,98.0,48,en,"Ariane Louis, Arthur Lang, Audrey Derrien, Flo...",Jad Charaf,"Horror, Thriller",None
20220,tt9907782,The Cursed,Eight for Silver,2021,6.2,22109.0,111,en,"Alistair Petrie, Amelia Crouch, Boyd Holbrook,...",Sean Ellis,"Fantasy, Horror, Mystery",https://image.tmdb.org/t/p/w500/bewmBcjJxHeipS...


### **2) FEATURE INGINEERING**

#### **a) High Value Features**

* **genres** : The #1 predictor. A user watching a "Comedy" wants another "Comedy."
* **director_names** (High Value): Directors carry a specific style/tone. 
* **actor_names** (High Value): For a general audience (families/seniors), "Star Power" matters. They might search "Jean Dujardin."
* **detected_language** (Specific to your Client): You should include this in the soup to group French movies together. This ensures a French movie is mathematically "closer" to other French movies than to Hollywood movies. 

#### **b) Filters for Ranking, Sorting**

* **averageRating**: To ensure you don't recommend bad movies (e.g., filter rating > 6.0).
* **numVotes**: To avoid recommending obscure movies with only 5 votes.
* **year** (startYear): To apply your "Nostalgia" logic (e.g., boost movies between 1980–2000).
* **runtimeMinutes**: Optional, but good for the user interface (e.g., "Short movies under 90 min").

#### **c) Features Extraction**

In [21]:
# Select only the features relevant for the Recommender System
features_df = meta_table[[
    'tconst',            # Keep ID to track the movie
    'primaryTitle',      # International Title (for search)
    'title',             # French Title (for display)
    'genres',            # SOUP INGREDIENT
    'director_name',    # SOUP INGREDIENT
    'actor_name',       # SOUP INGREDIENT
    'detected_language', # SOUP INGREDIENT
    'averageRating',     # FILTER
    'numVotes',          # FILTER
    'year',
    'runtimeMinutes'# FILTER
]].copy()

### **3) VECTORIZATION**

* Goal: Turn your 20,000+ movie "soups" into a giant matrix of numbers.
* Tool: CountVectorizer (better for keywords/names) or TfidfVectorizer (better for long plot descriptions). Since you are using metadata (Actors, Genres), use CountVectorizer.
* Optimization: Use stop_words='english' to remove noise, even though your content is French/International, names don't have stop words.

In [22]:
# CREATE THE "SOUP" (Feature Engineering) 
# # We mix all text features into one long string.
# We weight features by repeating them (Strategy from Presentation)
def create_soup(x):
    return (
        (str(x['actor_name']) + ' ') * 3 +  # Weight: 3x (Strongest Driver)
        (str(x['genres']) + ' ') * 2 +          # Weight: 2x (Habit Driver)
        (str(x['detected_language']) + ' ') * 2 + # Weight: 2x (French Preference)
        (str(x['director_name']) + ' ') * 2  +
        str(x['primaryTitle']) # Helping search accuracy
    ).lower()

features_df['soup'] = features_df.apply(create_soup, axis=1)

In [23]:
features_df

,tconst,primaryTitle,title,genres,director_name,actor_name,detected_language,averageRating,numVotes,year,runtimeMinutes,soup
0,tt0035423,Kate & Leopold,Kate et Léopold,"Comedy, Fantasy, Romance",James Mangold,"Bradley Whitford, Breckin Meyer, Hugh Jackman,...",en,6.4,93042.0,2001,118,"bradley whitford, breckin meyer, hugh jackman,..."
1,tt0036606,"Another Time, Another Place",Les Coeurs captifs,"Drama, War",Michael Radford,"Claudio Rosini, Denise Coffey, Gianluca Favill...",en,6.4,378.0,1983,118,"claudio rosini, denise coffey, gianluca favill..."
2,tt0048550,Rendez-vous of the Docks,Le rendez-vous des quais,Drama,Paul Carpita,"Albert Manach, André Maufray, Andrée Biancheri...",fr,6.6,69.0,1990,75,"albert manach, andré maufray, andrée biancheri..."
3,tt0065530,Le cercle des passions,Le cercle des passions,Drama,Claude d'Anna,"Assumpta Serna, Dora Calindri, Federico Pacifi...",fr,6.4,79.0,1983,108,"assumpta serna, dora calindri, federico pacifi..."
4,tt0067100,Firecracker,Attaque à mains nues,"Action, Drama, Thriller",Cirio H. Santiago,"Carolyn Smith, Chanda Romero, Darby Hinton, Do...",en,5.6,1019.0,1981,77,"carolyn smith, chanda romero, darby hinton, do..."
...,...,...,...,...,...,...,...,...,...,...,...,...
20217,tt9896916,Pilgrim's Progress,Le Voyage du pèlerin,"Adventure, Animation, Family",Robert Fernandez,"Andy Harrison, David Thorpe, Jasmine Jones, Jo...",en,6.5,1047.0,2019,108,"andy harrison, david thorpe, jasmine jones, jo..."
20218,tt9898858,Coffee & Kareem,Coffee & Kareem,"Action, Comedy, Crime",Michael Dowse,"Andrew Bachelor, Betty Gilpin, David Alan Grie...",en,5.2,15181.0,2020,88,"andrew bachelor, betty gilpin, david alan grie..."
20219,tt9904530,Scream Returns,Scream Returns,"Horror, Thriller",Jad Charaf,"Ariane Louis, Arthur Lang, Audrey Derrien, Flo...",en,5.7,98.0,2018,48,"ariane louis, arthur lang, audrey derrien, flo..."
20220,tt9907782,The Cursed,Eight for Silver,"Fantasy, Horror, Mystery",Sean Ellis,"Alistair Petrie, Amelia Crouch, Boyd Holbrook,...",en,6.2,22109.0,2021,111,"alistair petrie, amelia crouch, boyd holbrook,..."


In [24]:
# VECTORIZATION (Text to Numbers)
# We use CountVectorizer because we care about frequency of specific tags (Genre, Name).
# stop_words='english' removes "the", "and", etc.
count = CountVectorizer(stop_words='english', min_df=1)
count_matrix = count.fit_transform(features_df['soup'])
print(f"🔢 Matrix Shape: {count_matrix.shape} (Movies, Unique Words)")

🔢 Matrix Shape: (20222, 77446) (Movies, Unique Words)


### **4) MODEL: NearestNeighbors**

In [25]:
# TRAIN THE MODEL
# We use 'cosine' metric so it behaves exactly like cosine_similarity
# algorithm='brute' is actually often fastest for sparse matrices, but 'auto' is fine.
model_knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=20, n_jobs=-1)

# This "fit" doesn't compute the big matrix. It just indexes the data. (Fast & Low Memory)
model_knn.fit(count_matrix)

print("✅ Model Trained successfully")

✅ Model Trained successfully


In [26]:
# EXPORT
# Save the trained model and the matrix
out_dir = r"C:\Users\barba\Case_studies\Cinema_recommender\app"
with open(os.path.join(out_dir, 'knn_model.pkl'), 'wb') as f:
	pickle.dump(model_knn, f)
with open(os.path.join(out_dir, 'count_matrix.pkl'), 'wb') as f:
	pickle.dump(count_matrix, f)
with open(os.path.join(out_dir, 'movie_list.pkl'), 'wb') as f:
	pickle.dump(meta_table, f)

print("💾 Files saved for Streamlit!")

💾 Files saved for Streamlit!
